In [ ]:
# --- Repository setup ---
import os
from pathlib import Path

# Find the repository root (the folder that contains data/graphs) and work from there,
# so every relative path below resolves no matter where Jupyter was started.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'graphs').is_dir())
os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)


# Phase 2 — Two-LLM Baseline (mhgap_clinical_logic.md) vs Deterministic Traversal
This variant gives the Navigator LLM the module’s `mhgap_clinical_logic.md` (text rules) instead of graph relationships.

- Deterministic traversal still uses `mhgap_decision_graph.json` relationships internally.
- Two-LLM baseline NAV uses **clinical logic markdown + node catalog** and may drift/hallucinate.


## Config
- Graph JSON files are loaded from `PROJECT_ROOT` (mhGAP extracted graphs).
- User stories are loaded from `data/cases/virtual_patients`.

If you want to *reference code from another .ipynb*, note that notebooks are JSON files. You can read them with `json.loads(...)` and extract `cells[*].source`, but for reuse it is usually better to copy the needed helper functions into this notebook (or a `.py` module).


In [12]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

PROJECT_ROOT = REPO_ROOT / 'data' / 'graphs'
USERSTORY_ROOT = REPO_ROOT / 'data' / 'cases' / 'virtual_patients'
EVAL_OUTPUT_ROOT = REPO_ROOT / 'results' / 'raw_runs' / 'S3_llm_raw_text'
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT     :', PROJECT_ROOT)
print('USERSTORY_ROOT   :', USERSTORY_ROOT)
print('EVAL_OUTPUT_ROOT :', EVAL_OUTPUT_ROOT)


PROJECT_ROOT     : /Users/rakib/Downloads/mental-health-research/mental-health-research
USERSTORY_ROOT   : /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories
EVAL_OUTPUT_ROOT : /Users/rakib/Downloads/mental-health-research/detector_outputs/phase2_eval_runs/md


## Load mhGAP graphs
We build an index: `module_id -> graph_json`. This avoids hardcoding directory names.


In [13]:
def discover_graph_files(root: Path = PROJECT_ROOT) -> list[Path]:
    return sorted(root.rglob('mhgap_decision_graph.json'))


def load_graph(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def module_id(graph: dict[str, Any]) -> str:
    return str(graph.get('module', {}).get('id', 'UNKNOWN'))


def index_graphs_by_module() -> dict[str, dict[str, Any]]:
    idx: dict[str, dict[str, Any]] = {}
    for p in discover_graph_files(PROJECT_ROOT):
        g = load_graph(p)
        mid = module_id(g)
        if mid and mid != 'UNKNOWN':
            idx[mid] = g
    return idx


GRAPHS_BY_MODULE = index_graphs_by_module()
print('Indexed modules:', sorted(GRAPHS_BY_MODULE.keys()))


Indexed modules: ['CMH', 'DEM', 'DEP', 'EPI', 'OTH', 'PSY', 'SUB', 'SUI']


In [14]:
def discover_clinical_logic_files(root: Path = PROJECT_ROOT) -> list[Path]:
    return sorted(root.rglob('mhgap_clinical_logic.md'))


def load_text(path: Path) -> str:
    return path.read_text(encoding='utf-8')


def index_clinical_logic_by_module(graphs_by_module: dict[str, dict[str, Any]]) -> dict[str, str]:
    """Index mhgap_clinical_logic.md by module id.

    We map by folder: each module directory contains both mhgap_decision_graph.json and mhgap_clinical_logic.md.
    """
    idx: dict[str, str] = {}
    for md_path in discover_clinical_logic_files(PROJECT_ROOT):
        # try to find the sibling decision graph json to get module id
        graph_path = md_path.parent / 'mhgap_decision_graph.json'
        if not graph_path.exists():
            continue
        g = load_graph(graph_path)
        mid = module_id(g)
        if mid and mid != 'UNKNOWN':
            idx[mid] = load_text(md_path)
    return idx


CLINICAL_LOGIC_BY_MODULE = index_clinical_logic_by_module(GRAPHS_BY_MODULE)
print('Clinical logic modules:', sorted(CLINICAL_LOGIC_BY_MODULE.keys()))


Clinical logic modules: ['CMH', 'DEM', 'DEP', 'EPI', 'OTH', 'PSY', 'SUB', 'SUI']


## Load user-story cases
Each dataset JSON file under `USERSTORY_ROOT` has `{"cases": [...]}` where each case includes `user_story` and `node_sequence`.


In [15]:
def load_dataset(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def iter_dataset_files(root: Path = USERSTORY_ROOT) -> list[Path]:
    return sorted(root.rglob('*.json'))


# Example: list a few dataset files
dataset_files = iter_dataset_files(USERSTORY_ROOT)
print('Dataset files:', len(dataset_files))
for p in dataset_files[:5]:
    print('-', p)


Dataset files: 6
- /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/DEP/DEP_single_hop_single_hop_negative_1.json
- /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/DEP/DEP_single_hop_single_hop_positive_1.json
- /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/DEP/DEP_to_CMH_multi_hop_multi_hop_positive_1.json
- /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/DEP/DEP_to_OTH_multi_hop_multi_hop_positive_1.json
- /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/DEP/DEP_to_PSY_multi_hop_multi_hop_positive_1.json


## Models
- VP (virtual patient) model: answers YES/NO/NO_ANSWER from `user_story`
- NAV (navigator) model: chooses next node/question from graph context + previous Q/A logs

This notebook includes an OpenAI implementation (via `openai` SDK). If you add Gemini/others, keep the same request/response contract.


In [16]:
# %pip install -q openai


In [17]:
# Optional: %pip install -q openai
# Optional: %pip install -q google-genai

from openai import OpenAI

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY in your environment before running model calls')

client = OpenAI()

# Gemini (optional): only required if you include provider='gemini' in NAV_MODEL_SPECS
_GEMINI_CLIENT = None

VP_MODEL = 'gpt-4o-mini'

# Configure your NAV model sweep here (3 OpenAI + 3 Gemini).
# You can change these model ids without touching any other code.
NAV_MODEL_SPECS: list[dict[str, str]] = [
    {'provider': 'openai', 'model': 'gpt-4o'},
    {'provider': 'openai', 'model': 'gpt-4.1'},
    {'provider': 'openai', 'model': 'gpt-4o-mini'},
    {'provider': 'gemini', 'model': 'gemini-2.5-flash'},
    {'provider': 'gemini', 'model': 'gemini-2.5-pro'},
    {'provider': 'gemini', 'model': 'gemini-3.5-flash'},
]

Answer = Literal['YES', 'NO', 'NO_ANSWER']



## Helper functions: graph navigation
We keep a simple in-memory node index and outgoing-edge lookup.


In [18]:
def index_nodes(graph: dict[str, Any]) -> dict[str, dict[str, Any]]:
    return {str(n.get('id')): n for n in graph.get('nodes', []) if isinstance(n, dict) and n.get('id')}


def outgoing_edges(graph: dict[str, Any], from_id: str) -> list[dict[str, Any]]:
    rels = graph.get('relationships', [])
    out = []
    for r in rels:
        if isinstance(r, dict) and r.get('from') == from_id:
            out.append(r)
    return out


def find_start_node_id(graph: dict[str, Any]) -> str:
    for n in graph.get('nodes', []):
        if isinstance(n, dict) and n.get('is_starting') is True:
            return str(n.get('id'))
    raise ValueError('No is_starting node found')


def resolve_next_node_id_deterministic(graph: dict[str, Any], from_id: str, answer: str) -> str | None:
    # Deterministic: choose the first matching edge; if multiple, prefer stable ordering by `to`.
    candidates = [r for r in outgoing_edges(graph, from_id) if r.get('condition') == answer]
    if not candidates:
        return None
    candidates = sorted(candidates, key=lambda r: str(r.get('to', '')))
    return str(candidates[0].get('to'))


## Virtual patient LLM (VP): answer YES/NO/NO_ANSWER
VP only sees `user_story` and one question.


In [19]:
def vp_answer(user_story: str, question_text: str, *, model: str = VP_MODEL) -> tuple[Answer, dict[str, int]]:
    prompt = f"""You are a virtual patient.
Answer the clinician question using ONLY the story.
Return exactly one of: YES, NO, NO_ANSWER.
Use NO_ANSWER only if the story truly does not contain enough info.

PATIENT STORY:
{user_story}

QUESTION:
{question_text}
"""

    resp = client.responses.create(
        model=model,
        input=[
            {'role': 'developer', 'content': 'Output must be one token: YES, NO, or NO_ANSWER.'},
            {'role': 'user', 'content': prompt},
        ],
        temperature=0,
        max_output_tokens=25,
    )

    text = (resp.output_text or '').strip().upper()
    if text.startswith('YES'):
        ans: Answer = 'YES'
    elif text.startswith('NO_ANSWER') or text == 'NO ANSWER':
        ans = 'NO_ANSWER'
    elif text.startswith('NO'):
        ans = 'NO'
    else:
        ans = 'NO_ANSWER'

    u = getattr(resp, 'usage', None)
    usage = {
        'input_tokens': int(getattr(u, 'input_tokens', 0) or 0),
        'output_tokens': int(getattr(u, 'output_tokens', 0) or 0),
        'total_tokens': int(getattr(u, 'total_tokens', 0) or 0),
    }
    return ans, usage


## Navigator LLM (NAV): pick next node to ask
NAV sees the module `mhgap_clinical_logic.md` (text rules) + the node catalog + previous logs, and must output a `next_node_id` that exists in `nodes`.

NAV is intentionally *not* given graph relationships in this variant; it must infer the next step from the clinical-logic markdown and prior YES/NO/NEXT answers.


In [20]:
def _sanitize_tag(text: str) -> str:
    # filesystem-safe tag for filenames
    out = []
    for ch in str(text):
        if ch.isalnum() or ch in ('-', '_', '.'):
            out.append(ch)
        else:
            out.append('_')
    return ''.join(out)


def nav_model_tag(model_spec: dict[str, str]) -> str:
    provider = str(model_spec.get('provider') or 'unknown')
    model = str(model_spec.get('model') or 'unknown')
    return _sanitize_tag(f"{provider}-{model}")


def _nav_openai(payload: dict[str, Any], *, model: str) -> tuple[str, dict[str, int]]:
    resp = client.responses.create(
        model=model,
        input=[
            {'role': 'developer', 'content': 'Return only a JSON object with key next_node_id.'},
            {'role': 'user', 'content': json.dumps(payload, ensure_ascii=False)},
        ],
        temperature=0,
        # max_output_tokens=200,
    )

    u = getattr(resp, 'usage', None)
    usage = {
        'input_tokens': int(getattr(u, 'input_tokens', 0) or 0),
        'output_tokens': int(getattr(u, 'output_tokens', 0) or 0),
        'total_tokens': int(getattr(u, 'total_tokens', 0) or 0),
    }
    return (resp.output_text or '').strip(), usage


def _gemini_client():
    global _GEMINI_CLIENT
    if _GEMINI_CLIENT is not None:
        return _GEMINI_CLIENT
    api_key = os.getenv('GEMINI_API_KEY')
    if not api_key:
        raise RuntimeError('Set GEMINI_API_KEY in your environment before running Gemini model calls')
    from google import genai  # type: ignore

    _GEMINI_CLIENT = genai.Client(api_key=api_key)
    return _GEMINI_CLIENT


def _nav_gemini(payload: dict[str, Any], *, model: str) -> tuple[str, dict[str, int]]:
    c = _gemini_client()
    try:
        from google.genai import types  # type: ignore

        resp = c.models.generate_content(
            model=model,
            contents=[json.dumps(payload, ensure_ascii=False)],
            config=types.GenerateContentConfig(temperature=0),
        )
    except Exception:
        # Fallback if types module isn't available
        resp = c.models.generate_content(model=model, contents=[json.dumps(payload, ensure_ascii=False)])

    text = (getattr(resp, 'text', None) or '').strip()

    um = getattr(resp, 'usage_metadata', None)
    # usage_metadata may be an object or dict, depending on sdk version
    def _get(obj, key: str, default: int = 0) -> int:
        if obj is None:
            return default
        if isinstance(obj, dict):
            return int(obj.get(key, default) or default)
        return int(getattr(obj, key, default) or default)

    input_tokens = _get(um, 'prompt_token_count', 0)
    output_tokens = _get(um, 'candidates_token_count', 0)
    total = _get(um, 'total_token_count', input_tokens + output_tokens)

    usage = {'input_tokens': input_tokens, 'output_tokens': output_tokens, 'total_tokens': total}
    return text, usage


def nav_choose_next_node_id(
    *,
    clinical_logic_md: str,
    nodes: list[dict[str, Any]],
    current_node_id: str,
    step_logs: list[dict[str, Any]],
    model_spec: dict[str, str],
) -> tuple[str | None, dict[str, int]]:
    node_index = {str(n.get('id')): n for n in nodes if isinstance(n, dict) and n.get('id')}
    if current_node_id not in node_index:
        return None, {'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0}

    payload = {
        'mhgap_clinical_logic_md': clinical_logic_md,
        'nodes': nodes,
        'current_node_id': current_node_id,
        'previous_steps': step_logs,
        'rules': [
            'Choose the next node_id to ask from nodes only.',
            'Use the clinical logic markdown to decide the next step based on previous YES/NO/NEXT answers.',
            'Return only a JSON object with a single key next_node_id.',
            'Do not invent node ids that are not present in nodes.',
        ],
    }

    provider = str(model_spec.get('provider') or '')
    model = str(model_spec.get('model') or '')
    if not provider or not model:
        return None, {'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0}

    if provider == 'openai':
        raw, usage = _nav_openai(payload, model=model)
    elif provider == 'gemini':
        raw, usage = _nav_gemini(payload, model=model)
        print(raw)
    else:
        return None, {'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0}

    raw = (raw or '').strip()
    try:
        obj = json.loads(raw[raw.find('{') : raw.rfind('}') + 1])
        nxt = obj.get('next_node_id')
        if nxt is None:
            return None, usage
        nxt = str(nxt)
        if nxt not in node_index:
            return None, usage
        return nxt, usage
    except Exception:
        return None, usage



## Runners
We implement two runners with the same stop rules and the same step log format:
- `run_deterministic(...)`
- `run_two_llm_baseline(...)`


In [21]:
StopReason = Literal[
    'no_answer',
    'dead_end_no_edge',
    'exit_reached',
    'referral_reached_single_hop',
    'missing_target_module_graph',
    'missing_target_module_clinical_logic',
    'max_referral_hops',
    'max_iterations',
]


@dataclass
class RunResult:
    system: str
    module_id: str
    step_logs: list[dict[str, Any]]
    transitions: list[dict[str, Any]]
    stop_reason: StopReason
    stop_node_id: str
    usage: dict[str, Any]
    meta: dict[str, Any]


def run_deterministic(
    *,
    graph: dict[str, Any],
    user_story: str,
    max_iter: int = 20,
    graphs_by_module: dict[str, dict[str, Any]] | None = None,
    max_referral_hops: int = 1,
    verbose: bool = True,
) -> RunResult:
    root_module = module_id(graph)

    active_graph = graph
    node_idx = index_nodes(active_graph)
    start_id = find_start_node_id(active_graph)

    current = start_id
    referral_hops = 0
    step_logs: list[dict[str, Any]] = []
    transitions: list[dict[str, Any]] = []
    usage = {
        'vp': {'calls': 0, 'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0},
        'nav': {'calls': 0, 'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0},
    }
    meta = {
        'vp_model': VP_MODEL,
        'nav_model_spec': None,
    }

    for _ in range(max_iter):
        node = node_idx.get(current)
        if not node:
            return RunResult('deterministic', root_module, step_logs, transitions, 'dead_end_no_edge', current, usage, meta)

        node_type = node.get('type')
        log_entry = {
            'id': node.get('id'),
            'type': node_type,
            'is_starting': bool(node.get('is_starting', False)),
            'text': node.get('text'),
        }

        if node_type == 'assessment':
            ans, u = vp_answer(user_story, str(node.get('text', '')))
            usage['vp']['calls'] += 1
            usage['vp']['input_tokens'] += u.get('input_tokens', 0)
            usage['vp']['output_tokens'] += u.get('output_tokens', 0)
            usage['vp']['total_tokens'] += u.get('total_tokens', 0)
            log_entry['user_answer'] = ans
            step_logs.append(log_entry)
            if verbose:
                print(json.dumps(log_entry, ensure_ascii=False))

            if ans == 'NO_ANSWER':
                return RunResult('deterministic', root_module, step_logs, transitions, 'no_answer', current, usage, meta)

            nxt = resolve_next_node_id_deterministic(active_graph, current, ans)
            if not nxt:
                return RunResult('deterministic', root_module, step_logs, transitions, 'dead_end_no_edge', current, usage, meta)
            transitions.append({'from': current, 'answer': ans, 'to': nxt})
            current = nxt

        elif node_type == 'exit':
            step_logs.append(log_entry)
            if verbose:
                print(json.dumps(log_entry, ensure_ascii=False))
            return RunResult('deterministic', root_module, step_logs, transitions, 'exit_reached', current, usage, meta)

        elif node_type == 'referral':
            log_entry['target_module'] = node.get('target_module')
            step_logs.append(log_entry)
            if verbose:
                print(json.dumps(log_entry, ensure_ascii=False))

            if graphs_by_module is None:
                return RunResult('deterministic', root_module, step_logs, transitions, 'referral_reached_single_hop', current, usage, meta)

            target_module = str(node.get('target_module') or '')
            if not target_module or target_module not in graphs_by_module:
                return RunResult('deterministic', root_module, step_logs, transitions, 'missing_target_module_graph', current, usage, meta)

            if referral_hops >= max_referral_hops:
                return RunResult('deterministic', root_module, step_logs, transitions, 'max_referral_hops', current, usage, meta)

            from_id = current
            referral_hops += 1
            active_graph = graphs_by_module[target_module]
            node_idx = index_nodes(active_graph)
            current = find_start_node_id(active_graph)
            transitions.append({'from': from_id, 'answer': 'REFERRAL', 'to': current, 'target_module': target_module})

        else:
            step_logs.append(log_entry)
            if verbose:
                print(json.dumps(log_entry, ensure_ascii=False))
            nxt = resolve_next_node_id_deterministic(active_graph, current, 'NEXT')
            if not nxt:
                return RunResult('deterministic', root_module, step_logs, transitions, 'dead_end_no_edge', current, usage, meta)
            transitions.append({'from': current, 'answer': 'NEXT', 'to': nxt})
            current = nxt

    return RunResult('deterministic', root_module, step_logs, transitions, 'max_iterations', current, usage, meta)


def run_two_llm_baseline(
    *,
    graph: dict[str, Any],
    clinical_logic_md: str,
    user_story: str,
    nav_model_spec: dict[str, str],
    max_iter: int = 20,
    verbose: bool = True,
    graphs_by_module: dict[str, dict[str, Any]] | None = None,
    clinical_logic_by_module: dict[str, str] | None = None,
    max_referral_hops: int = 1,
) -> RunResult:
    """Two-LLM baseline.

    - Single-hop behavior: pass graphs_by_module=None (default) to stop at referral nodes.
    - Multi-hop behavior: pass graphs_by_module=GRAPHS_BY_MODULE AND clinical_logic_by_module=CLINICAL_LOGIC_BY_MODULE to follow referrals.
    """
    root_module = module_id(graph)

    active_graph = graph
    active_logic_md = clinical_logic_md
    node_idx = index_nodes(active_graph)
    start_id = find_start_node_id(active_graph)

    current = start_id
    referral_hops = 0
    step_logs: list[dict[str, Any]] = []
    transitions: list[dict[str, Any]] = []
    usage = {
        'vp': {'calls': 0, 'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0},
        'nav': {'calls': 0, 'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0},
    }
    meta = {
        'vp_model': VP_MODEL,
        'nav_model_spec': nav_model_spec,
    }

    for _ in range(max_iter):
        node = node_idx.get(current)
        if not node:
            return RunResult('two_llm_baseline', root_module, step_logs, transitions, 'dead_end_no_edge', current, usage, meta)

        node_type = node.get('type')
        log_entry = {
            'id': node.get('id'),
            'type': node_type,
            'is_starting': bool(node.get('is_starting', False)),
            'text': node.get('text'),
        }

        if node_type == 'assessment':
            ans, u = vp_answer(user_story, str(node.get('text', '')))
            usage['vp']['calls'] += 1
            usage['vp']['input_tokens'] += u.get('input_tokens', 0)
            usage['vp']['output_tokens'] += u.get('output_tokens', 0)
            usage['vp']['total_tokens'] += u.get('total_tokens', 0)
            log_entry['user_answer'] = ans
            step_logs.append(log_entry)
            if verbose:
                print(json.dumps(log_entry, ensure_ascii=False))

            if ans == 'NO_ANSWER':
                return RunResult('two_llm_baseline', root_module, step_logs, transitions, 'no_answer', current, usage, meta)

            nxt, u_nav = nav_choose_next_node_id(
                clinical_logic_md=active_logic_md,
                nodes=active_graph.get('nodes', []),
                current_node_id=current,
                step_logs=step_logs,
                model_spec=nav_model_spec,
            )
            usage['nav']['calls'] += 1
            usage['nav']['input_tokens'] += u_nav.get('input_tokens', 0)
            usage['nav']['output_tokens'] += u_nav.get('output_tokens', 0)
            usage['nav']['total_tokens'] += u_nav.get('total_tokens', 0)
            if not nxt:
                return RunResult('two_llm_baseline', root_module, step_logs, transitions, 'dead_end_no_edge', current, usage, meta)

            transitions.append({'from': current, 'answer': ans, 'to': nxt})
            current = nxt

        elif node_type == 'exit':
            step_logs.append(log_entry)
            if verbose:
                print(json.dumps(log_entry, ensure_ascii=False))
            return RunResult('two_llm_baseline', root_module, step_logs, transitions, 'exit_reached', current, usage, meta)

        elif node_type == 'referral':
            log_entry['target_module'] = node.get('target_module')
            step_logs.append(log_entry)
            if verbose:
                print(json.dumps(log_entry, ensure_ascii=False))

            if graphs_by_module is None:
                return RunResult('two_llm_baseline', root_module, step_logs, transitions, 'referral_reached_single_hop', current, usage, meta)

            target_module = str(node.get('target_module') or '')
            if not target_module or target_module not in graphs_by_module:
                return RunResult('two_llm_baseline', root_module, step_logs, transitions, 'missing_target_module_graph', current, usage, meta)

            if not clinical_logic_by_module or target_module not in clinical_logic_by_module:
                return RunResult('two_llm_baseline', root_module, step_logs, transitions, 'missing_target_module_clinical_logic', current, usage, meta)

            if referral_hops >= max_referral_hops:
                return RunResult('two_llm_baseline', root_module, step_logs, transitions, 'max_referral_hops', current, usage, meta)

            from_id = current
            referral_hops += 1
            active_graph = graphs_by_module[target_module]
            active_logic_md = clinical_logic_by_module[target_module]
            node_idx = index_nodes(active_graph)
            current = find_start_node_id(active_graph)
            transitions.append({'from': from_id, 'answer': 'REFERRAL', 'to': current, 'target_module': target_module})

        else:
            step_logs.append(log_entry)
            if verbose:
                print(json.dumps(log_entry, ensure_ascii=False))
            nxt, u_nav = nav_choose_next_node_id(
                clinical_logic_md=active_logic_md,
                nodes=active_graph.get('nodes', []),
                current_node_id=current,
                step_logs=step_logs,
                model_spec=nav_model_spec,
            )
            usage['nav']['calls'] += 1
            usage['nav']['input_tokens'] += u_nav.get('input_tokens', 0)
            usage['nav']['output_tokens'] += u_nav.get('output_tokens', 0)
            usage['nav']['total_tokens'] += u_nav.get('total_tokens', 0)
            if not nxt:
                return RunResult('two_llm_baseline', root_module, step_logs, transitions, 'dead_end_no_edge', current, usage, meta)
            transitions.append({'from': current, 'answer': 'NEXT', 'to': nxt})
            current = nxt

    return RunResult('two_llm_baseline', root_module, step_logs, transitions, 'max_iterations', current, usage, meta)



## Example: run one case
Select a dataset JSON (with user_story) and run both systems against the module graph.


In [22]:
def compare_to_expected(node_sequence: list[dict[str, Any]], step_logs: list[dict[str, Any]]) -> dict[str, Any]:
    # Compare overall node-id sequence
    expected_ids = [n.get('id') for n in node_sequence if isinstance(n, dict) and n.get('id')]
    actual_ids = [s.get('id') for s in step_logs if isinstance(s, dict) and s.get('id')]

    exact_path_match = actual_ids == expected_ids
    prefix_path_match = actual_ids == expected_ids[: len(actual_ids)]

    # Compare assessment findings: expected condition_met vs actual user_answer
    expected_findings = {
        n.get('id'): n.get('condition_met')
        for n in node_sequence
        if isinstance(n, dict) and n.get('type') == 'assessment' and n.get('condition_met') in ('YES', 'NO')
    }

    actual_findings = {
        s.get('id'): s.get('user_answer')
        for s in step_logs
        if isinstance(s, dict) and s.get('type') == 'assessment' and s.get('user_answer') in ('YES', 'NO')
    }

    total = len(expected_findings)
    matched = 0
    mismatches = []
    missing = []

    for nid, exp in expected_findings.items():
        got = actual_findings.get(nid)
        if got is None:
            missing.append({'id': nid, 'expected': exp, 'actual': None})
            continue
        if got == exp:
            matched += 1
        else:
            mismatches.append({'id': nid, 'expected': exp, 'actual': got})

    finding_accuracy = (matched / total) if total else None

    # Extra findings predicted that aren't in expected assessment list
    extra = [
        {'id': nid, 'expected': None, 'actual': ans}
        for nid, ans in actual_findings.items()
        if nid not in expected_findings
    ]

    return {
        'expected_ids': expected_ids,
        'actual_ids': actual_ids,
        'exact_path_match': exact_path_match,
        'prefix_path_match': prefix_path_match,
        'finding_accuracy': finding_accuracy,
        'missing_findings': missing,
        'mismatched_findings': mismatches,
        'extra_findings': extra,
        'counts': {
            'expected_assessment_findings': total,
            'matched': matched,
            'missing': len(missing),
            'mismatched': len(mismatches),
            'extra': len(extra),
        },
    }


def total_tokens(usage: dict[str, Any]) -> int:
    return int(usage.get('vp', {}).get('total_tokens', 0) or 0) + int(usage.get('nav', {}).get('total_tokens', 0) or 0)


In [32]:
# Run every dataset JSON under USERSTORY_ROOT / 'DEP' with multiple NAV models (3 OpenAI + 3 Gemini).
# This removes the need to manually change dataset_path filenames.

DEP_DIR = USERSTORY_ROOT / 'SUI'
dataset_files = sorted(DEP_DIR.glob('*.json'))
print('DEP dataset files:', len(dataset_files))
for p in dataset_files:
    print('-', p.name)

# Controls
LIMIT_CASES_PER_FILE: int | None = None
VERBOSE_STEP_LOGS = False  # set True to print every step json
MAX_REFERRAL_HOPS = 1

for dataset_path in dataset_files:
    payload = load_dataset(dataset_path)
    cases = payload.get('cases', []) or []
    if not isinstance(cases, list):
        continue

    for case_index, case0 in enumerate(cases):
        if LIMIT_CASES_PER_FILE is not None and case_index >= LIMIT_CASES_PER_FILE:
            break

        cid = str(case0.get('case_id') or f'case{case_index}')
        stem = dataset_path.stem

        source_module = str(case0.get('source_module') or '')
        if source_module not in GRAPHS_BY_MODULE or source_module not in CLINICAL_LOGIC_BY_MODULE:
            continue

        graph = GRAPHS_BY_MODULE[source_module]
        clinical_logic_md = CLINICAL_LOGIC_BY_MODULE[source_module]
        story = str(case0.get('user_story', ''))
        expected_seq = case0.get('node_sequence', [])

        enable_multi_hop = str(case0.get('category', '')).startswith('multi_hop')
        graphs_by_module = GRAPHS_BY_MODULE if enable_multi_hop else None
        clinical_logic_by_module = CLINICAL_LOGIC_BY_MODULE if enable_multi_hop else None

        # Deterministic run (once per case)
        # det = run_deterministic(
        #     graph=graph,
        #     user_story=story,
        #     graphs_by_module=graphs_by_module,
        #     max_referral_hops=MAX_REFERRAL_HOPS,
        #     verbose=VERBOSE_STEP_LOGS,
        # )
        # det_eval = compare_to_expected(expected_seq, det.step_logs)
        # det_path = EVAL_OUTPUT_ROOT / f'{stem}__{cid}__det.json'
        # det_payload = {
        #     **det.__dict__,
        #     'dataset_path': str(dataset_path),
        #     'case_id': cid,
        #     'expected_node_sequence': expected_seq,
        #     'evaluation': det_eval,
        #     'total_tokens': total_tokens(det.usage),
        # }
        # det_path.write_text(json.dumps(det_payload, indent=2, ensure_ascii=False), encoding='utf-8')

        # Baseline runs (one per NAV model)
     
        for nav_spec in NAV_MODEL_SPECS:
            tag = nav_model_tag(nav_spec)
            base = run_two_llm_baseline(
                graph=graph,
                clinical_logic_md=clinical_logic_md,
                clinical_logic_by_module=clinical_logic_by_module,
                user_story=story,
                nav_model_spec=nav_spec,
                graphs_by_module=graphs_by_module,
                max_referral_hops=MAX_REFERRAL_HOPS,
                verbose=VERBOSE_STEP_LOGS,
            )
            base_eval = compare_to_expected(expected_seq, base.step_logs)
            base_path = EVAL_OUTPUT_ROOT / f'{stem}__{cid}__base__nav-{tag}.json'
            base_payload = {
                **base.__dict__,
                'dataset_path': str(dataset_path),
                'case_id': cid,
                'nav_model_spec': nav_spec,
                'expected_node_sequence': expected_seq,
                'evaluation': base_eval,
                'total_tokens': total_tokens(base.usage),
            }
            base_path.write_text(json.dumps(base_payload, indent=2, ensure_ascii=False), encoding='utf-8')

        print('Saved case:', dataset_path.name, cid)



DEP dataset files: 3
- SUI_single_hop_single_hop_negative_1.json
- SUI_single_hop_single_hop_positive_1.json
- SUI_to_OTH_multi_hop_multi_hop_positive_1.json
```json
{"next_node_id": "SUI_A2"}
```
```json
{"next_node_id": "SUI_A4"}
```
```json
{"next_node_id": "SUI_A5"}
```
```json
{"next_node_id": "SUI_A6"}
```
```json
{"next_node_id": "SUI_M7"}
```
```json
{"next_node_id": "SUI_E1"}
```
```json
{
  "next_node_id": "SUI_A2"
}
```
```json
{
  "next_node_id": "SUI_A3"
}
```
```json
{"next_node_id": "SUI_D3"}
```
```json
{
  "next_node_id": "SUI_A4"
}
```
```json
{"next_node_id": "SUI_A5"}
```
```json
{"next_node_id": "SUI_A6"}
```
```json
{
  "next_node_id": "SUI_M7"
}
```
```json
{
  "next_node_id": "SUI_E1"
}
```
{
  "next_node_id": "SUI_A2"
}
{"next_node_id": "SUI_A3"}
```json
{
  "next_node_id": "SUI_D3"
}
```
{"next_node_id": "SUI_A4"}
```json
{
  "next_node_id": "SUI_A5"
}
```
{"next_node_id": "SUI_A6"}
```json
{
  "next_node_id": "SUI_M7"
}
```
```json
{
  "next_node_id": "SUI_E1